In [0]:
# ===========================================
# Notebook Name:
# 02_build_tournament_results
#
# Purpose:
# Parse raw Pokemon tournament result JSON
# and build the normalized Silver ranking
# table. (Tournament master data itself is
# built separately by 00_build_tournaments
# from bronze.tournament_list_raw.)
#
# Input:
# pokemon.bronze.event_result_raw
#
# Output:
# pokemon.silver.tournament_results
#
# Merge key:
# tournament_id, rank
# ===========================================

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

EVENT_RESULT_RAW_TABLE = (
    "pokemon.bronze.event_result_raw"
)

TOURNAMENT_RESULTS_TABLE = (
    "pokemon.silver.tournament_results"
)

print("Input :", EVENT_RESULT_RAW_TABLE)
print("Output:", TOURNAMENT_RESULTS_TABLE)

In [0]:
bronze_df = spark.table(
    EVENT_RESULT_RAW_TABLE
)

display(
    bronze_df.select(
        "tournament_id",
        "http_status",
        "response_hash",
        "scraped_at",
        "ingestion_date",
        F.length("payload").alias(
            "payload_length"
        ),
    )
    .orderBy(F.col("scraped_at").desc())
)

In [0]:
latest_window = (
    Window
    .partitionBy("tournament_id")
    .orderBy(
        F.col("scraped_at").desc(),
        F.col("response_hash").desc(),
    )
)

latest_raw_df = (
    bronze_df
    .withColumn(
        "row_number",
        F.row_number().over(latest_window),
    )
    .filter(F.col("row_number") == 1)
    .drop("row_number")
)

display(
    latest_raw_df.select(
        "tournament_id",
        "response_hash",
        "scraped_at",
    )
)

In [0]:
sample_row = (
    latest_raw_df
    .select("payload")
    .where(F.col("payload").isNotNull())
    .limit(1)
    .first()
)

if sample_row is None:
    raise ValueError(
        "No parseable payload found "
        "in event_result_raw"
    )

sample_payload = sample_row["payload"]

print(
    "Sample payload length:",
    len(sample_payload),
)
print(sample_payload[:500])

In [0]:
json_schema = (
    spark.range(1)
    .select(
        F.schema_of_json(
            F.lit(sample_payload)
        ).alias("json_schema")
    )
    .first()["json_schema"]
)

print(json_schema)

In [0]:
parsed_df = (
    latest_raw_df
    .withColumn(
        "parsed_payload",
        F.from_json(
            F.col("payload"),
            json_schema,
        ),
    )
    .select(
        "tournament_id",
        "source_url",
        "response_hash",
        "scraped_at",
        F.col(
            "parsed_payload.code"
        ).alias("api_code"),
        F.col(
            "parsed_payload.count"
        ).alias("result_count"),
        F.col(
            "parsed_payload.results"
        ).alias("results"),
    )
)

parsed_df.printSchema()
display(parsed_df)

In [0]:
exploded_results_df = (
    parsed_df
    .select(
        "tournament_id",
        "source_url",
        "response_hash",
        "scraped_at",
        F.explode_outer(
            "results"
        ).alias("result"),
    )
    .withColumn(
        "result_json",
        F.to_json("result"),
    )
)

display(exploded_results_df)

In [0]:
tournament_results_df = (
    exploded_results_df
    .select(
        F.col("tournament_id"),
        F.get_json_object(
            "result_json",
            "$.rank",
        ).cast("int").alias("rank"),
        F.get_json_object(
            "result_json",
            "$.player_id",
        ).alias("player_id"),
        F.get_json_object(
            "result_json",
            "$.name",
        ).alias("player_name"),
        F.get_json_object(
            "result_json",
            "$.area",
        ).alias("area"),
        F.get_json_object(
            "result_json",
            "$.point",
        ).cast("int").alias("point"),
        F.get_json_object(
            "result_json",
            "$.show_profile",
        ).cast("boolean").alias(
            "show_profile"
        ),
        F.get_json_object(
            "result_json",
            "$.deck_id",
        ).alias("deck_code"),
        F.col("source_url"),
        F.col("response_hash"),
        F.col("scraped_at").alias(
            "source_scraped_at"
        ),
        F.current_timestamp().alias(
            "updated_at"
        ),
    )
    .withColumn(
        "deck_url",
        F.when(
            F.col("deck_code").isNotNull(),
            F.concat(
                F.lit(
                    "https://www.pokemon-card.com/"
                    "deck/confirm.html/deckID/"
                ),
                F.col("deck_code"),
            ),
        ),
    )
    .withColumn(
        "deck_print_url",
        F.when(
            F.col("deck_code").isNotNull(),
            F.concat(
                F.lit(
                    "https://www.pokemon-card.com/"
                    "deck/print.html/deckID/"
                ),
                F.col("deck_code"),
                F.lit("/"),
            ),
        ),
    )
    .filter(
        F.col("rank").isNotNull()
    )
)

display(
    tournament_results_df.orderBy(
        "tournament_id",
        "rank",
    )
)

In [0]:
display(
    tournament_results_df
    .groupBy("tournament_id")
    .agg(
        F.count("*").alias(
            "result_rows"
        ),
        F.countDistinct(
            "player_id"
        ).alias("unique_players"),
        F.countDistinct(
            "deck_code"
        ).alias("unique_decks"),
        F.sum(
            F.when(
                F.col(
                    "deck_code"
                ).isNull(),
                1,
            ).otherwise(0)
        ).alias("missing_deck_count"),
        F.min("rank").alias("min_rank"),
        F.max("rank").alias("max_rank"),
    )
)

In [0]:
# The results API is fetched as a single
# page (see 01_ingest_event_results). A
# mismatch here for large tournaments is
# expected truncation, not a parse bug, so
# this is informational only.
validation_df = (
    parsed_df
    .select(
        "tournament_id",
        F.col(
            "result_count"
        ).cast("int").alias(
            "expected_count"
        ),
    )
    .join(
        tournament_results_df
        .groupBy("tournament_id")
        .agg(
            F.count("*").alias(
                "actual_count"
            )
        ),
        on="tournament_id",
        how="left",
    )
    .withColumn(
        "is_valid",
        F.col("expected_count")
        == F.col("actual_count"),
    )
)

display(
    validation_df.filter(
        ~F.col("is_valid")
    )
)

In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS
{TOURNAMENT_RESULTS_TABLE}
(
    tournament_id STRING NOT NULL,
    rank INT NOT NULL,
    player_id STRING,
    player_name STRING,
    area STRING,
    point INT,
    show_profile BOOLEAN,
    deck_code STRING,
    source_url STRING,
    response_hash STRING,
    source_scraped_at TIMESTAMP,
    updated_at TIMESTAMP,
    deck_url STRING,
    deck_print_url STRING,
    deck_hash STRING,
    deck_hash_version STRING
)
USING DELTA
COMMENT
'Normalized Pokemon tournament ranking '
'and deck results'
""")

In [0]:
duplicate_rank_df = (
    tournament_results_df
    .groupBy(
        "tournament_id",
        "rank",
    )
    .count()
    .filter(F.col("count") > 1)
)

duplicate_rank_count = (
    duplicate_rank_df.count()
)

if duplicate_rank_count > 0:
    display(duplicate_rank_df)

    raise ValueError(
        f"{duplicate_rank_count} duplicate "
        "(tournament_id, rank) pairs detected"
    )

print(
    "Validation passed: "
    "one row per (tournament_id, rank)"
)

In [0]:
tournament_results_df.createOrReplaceTempView(
    "staged_tournament_results"
)

spark.sql(f"""
MERGE INTO
{TOURNAMENT_RESULTS_TABLE} AS target
USING
staged_tournament_results AS source
ON  target.tournament_id = source.tournament_id
AND target.rank = source.rank

WHEN MATCHED THEN UPDATE SET
    target.player_id = source.player_id,
    target.player_name = source.player_name,
    target.area = source.area,
    target.point = source.point,
    target.show_profile = source.show_profile,
    target.deck_code = source.deck_code,
    target.source_url = source.source_url,
    target.response_hash = source.response_hash,
    target.source_scraped_at =
        source.source_scraped_at,
    target.updated_at = source.updated_at,
    target.deck_url = source.deck_url,
    target.deck_print_url =
        source.deck_print_url

WHEN NOT MATCHED THEN INSERT (
    tournament_id,
    rank,
    player_id,
    player_name,
    area,
    point,
    show_profile,
    deck_code,
    source_url,
    response_hash,
    source_scraped_at,
    updated_at,
    deck_url,
    deck_print_url
)
VALUES (
    source.tournament_id,
    source.rank,
    source.player_id,
    source.player_name,
    source.area,
    source.point,
    source.show_profile,
    source.deck_code,
    source.source_url,
    source.response_hash,
    source.source_scraped_at,
    source.updated_at,
    source.deck_url,
    source.deck_print_url
)
""")

print("Tournament result table merged.")

In [0]:
display(
    spark.table(
        TOURNAMENT_RESULTS_TABLE
    )
    .orderBy(
        F.col(
            "source_scraped_at"
        ).desc(),
        "tournament_id",
        "rank",
    )
    .limit(50)
)